# Assignment 6: Diffusion models

In this notebook, you will build, train, and sample from a Denoising Diffusion Probabilistic Model (DDPM). Our goal is to generate images of handwritten digits by training our model on the MNIST dataset.

This is a hands-on exercise. You will be guided through the concepts, and then you will write the necessary Python code to make the model run.

We will cover:

-   Setup & Data Loading: Preparing our environment and the MNIST dataset.
-    The Forward Process (Noising): A deep dive into the theory and implementation of gradually adding noise to images.
-    The Model (U-Net): Building the neural network that will learn to reverse the noising process.
-    The Training Loop: Writing the code to train our model to predict the added noise.
-    The Reverse Process (Sampling): Using our trained model to generate new images from pure noise.

### Useful links

- [Step by Step Visual Introduction to Diffusion Models](https://medium.com/@kemalpiro/step-by-step-visual-introduction-to-diffusion-models-235942d2f15c)
- [Stable Diffusion Clearly Explained!](https://codoraven.com/blog/ai/stable-diffusion-clearly-explained/)
- [How diffusion models work: the math from scratch](https://theaisummer.com/diffusion-models/)
- [What are Diffusion Models?](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)
- [Denoising DIffusion Probabilistic Models (original paper)](https://arxiv.org/abs/2006.11239)

<div style="text-align: center;">
<img src="https://www.siam.org/media/agxdzywa/figure1.jpg" width="800" style="center"/>
</div>

## 1. Setup and Configuration

First, you need to set up your environment by importing the necessary libraries i.e. _numpy, matplotlib and pytorch_. Then, define the main configuration parameters for our experiment. Keeping them in one place makes it easy to adjust them later.

Set the device to "cuda" if a GPU is available, otherwise use "cpu".

Define the following configuration variables:
-  __IMG_SIZE = 32__
-  __BATCH_SIZE = 128__
-  __EPOCHS = 10__
-  __LEARNING_RATE = 2e-4__
-  __TIMESTEPS = 1000__

If you are working without a GPU, you may want to adjust these settings to your system.


In [ ]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

IMG_SIZE = 32
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 2e-4
TIMESTEPS = 1000


## 2. Data Preparation for PyTorch

With the setup complete, you need to load the MNIST dataset. You'll apply a few important transformations to the images:

1.    **Resize**: Resize the images from their original 28x28 to 32x32 pixels. This is often done to make the U-Net architecture, which typically uses powers of two for downsampling.
2.    **ToTensor**: Convert the images into PyTorch tensors.
3.    **Normalize**: Scale the pixel values from the standard [0, 1] range to [-1, 1]. This is a crucial step that centers the data around zero, which better matches the distribution of the Gaussian noise we'll be adding.

**Your Task:**

-    Create a `transforms.Compose` sequence with the three transformations described above.
-    Use `torchvision.datasets.MNIST` to load the training data, applying your transformation sequence.
-    Wrap the dataset in a `DataLoader` for efficient batching.
-    Write a helper function to visualize a grid of images from a batch. Remember to un-normalize the images from `[-1, 1]` back to `[0, 1]` before displaying them.
-    Call your function to display a batch of images to verify everything is working.


In [ ]:
# TODO: create the transform pipeline for MNIST
transform = transforms.Compose([
    ...,
])

# TODO: load the MNIST training dataset
train_dataset = ...

# TODO: create the dataloader
train_loader = ...


def show_images(images, title=None, nrow=8):
    # TODO: un-normalize images from [-1, 1] to [0, 1] before plotting
    images = ...
    grid = make_grid(images, nrow=nrow)

    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
    plt.axis("off")
    if title is not None:
        plt.title(title)
    plt.show()


# TODO: display one batch to verify the preprocessing
images, labels = ...
show_images(...)


## 3. The Forward Diffusion Process (Noising)

<div style="text-align: center;">
<img src="https://codoraven.com/app/uploads/2023/02/02_diffusion_model_overview-768x508.jpg" width="800" style="center"/>
</div>

The forward process, `q`, is where we systematically destroy information in an image by adding Gaussian noise over `T` timesteps. The forward process is stochastic, but its variance at each step is controlled by a fixed, predefined **variance schedule**.

Let's break down the core terms:

*   **Beta (`beta_t`): The Variance Schedule.** This is a sequence of small constants that determine *how much* noise is added at each timestep `t`:

$$\beta_1, \beta_2, ..., \beta_T$$

We will use a **linear schedule** where beta increases from a small value to a larger one. For this exercise, we'll use `beta_start = 0.0001` and `beta_end = 0.02`. A small beta at the beginning means we add very little noise, and a larger beta at the end means we add more noise.

*   **Alpha (`alpha_t`): The Signal Rate.** This is defined as:

$$\alpha_t = 1 - \beta_t$$

It represents how much of the image signal from the previous step is preserved. If `beta_t` is small, `alpha_t` is close to 1, meaning most of the signal is kept.

*   **Alpha-bar (`alpha_bar_t`): The Cumulative Signal Rate.** This is the cumulative product of all alpha values up to timestep `t`:

$$\bar{\alpha}_t = \alpha_1 \cdot \alpha_2 \cdot ... \cdot \alpha_t$$

This term is incredibly important because it allows us to calculate the noised image `x_t` directly from the original image `x_0` in a single step, without having to iterate `t` times. It represents the total amount of signal from the original image `x_0` that is still present at timestep `t`.

This leads us to the "closed-form" formula for the forward process:

$$ x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon $$

Here, `x_0` is your original image and `epsilon` is pure Gaussian noise (`torch.randn_like(x_0)`). This elegant equation tells us that any noised image `x_t` is just a mix of two components: the original image `x_0`, scaled down by the square root of `alpha_bar_t`, and pure noise `epsilon`, scaled up by the square root of `1 - alpha_bar_t`. The scaling factors are chosen so that the variance of the output is preserved, which helps stabilize training.

**Your Task:**
1.  Define a function that creates a linear schedule for `betas` from `0.0001` to `0.02` over `TIMESTEPS`.
2.  Calculate `alphas` (`1 - betas`).
3.  Calculate `alphas_cumprod` (the cumulative product of `alphas`).
4.  Implement the `forward_diffusion_sample` function. It should take an original image `x_0` and a tensor of timesteps `t` as input. Inside the function, you will:
    *   Generate random noise `epsilon`.
    *   Look up the correct `alpha_bar_t` value for each image in the batch using its corresponding timestep `t`.
    *   Reshape the selected `alpha_bar_t` values so they broadcast across image tensors of shape `(batch, channels, height, width)`.
    *   Use the formula above to compute and return the noised image `x_t` and the noise `epsilon` that was used.
5.  Use your function to visualize the forward process. Take one image from your dataset and show what it looks like at several different timesteps (e.g., `t = 0, 100, 250, 500, 999`).


In [ ]:
def linear_beta_schedule(timesteps, start=0.0001, end=0.02):
    # TODO: return a tensor of betas with length timesteps
    return ...


# TODO: create betas, alphas, and alphas_cumprod on DEVICE
betas = ...
alphas = ...
alphas_cumprod = ...


def forward_diffusion_sample(x_0, t):
    # TODO: sample Gaussian noise with the same shape as x_0
    noise = ...

    # TODO: gather alpha_bar values for each item in the batch and reshape for broadcasting
    alpha_bar_t = ...

    # TODO: apply the closed-form forward diffusion formula
    x_t = ...
    return x_t, noise


# TODO: visualize one training image at several timesteps
sample_image = ...
timesteps_to_show = [0, 100, 250, 500, 999]
noisy_versions = []

for step in timesteps_to_show:
    t = torch.tensor([step], device=DEVICE)
    noisy_image, _ = forward_diffusion_sample(sample_image, t)
    noisy_versions.append(noisy_image.squeeze(0).cpu())

show_images(torch.stack(noisy_versions), title="Forward diffusion", nrow=len(timesteps_to_show))


## 4. The U-Net Model (The Denoising Network)


<div style="text-align: center;">
<img src="https://codoraven.com/app/uploads/2023/02/08_architecture_dm.jpg" width="800" style="center"/>
</div>

Now for the brain of our operation. We need a model that can reverse the noising process. This model will take a noisy image `x_t` and the timestep `t` as input, and its job is to predict the noise `epsilon` that was added to it. A **U-Net** architecture is the standard choice for this task due to its effectiveness in image-to-image tasks.

**Sinusoidal time embeddings**

The model also needs to know which diffusion timestep it is denoising. A common way to represent the integer timestep `t` is with sinusoidal embeddings, similar to positional encodings in Transformers. For an embedding dimension `d`, use pairs of sine and cosine values:

$$
\operatorname{emb}(t)_{2i} = \sin\left(\frac{t}{10000^{2i/d}}\right), \qquad
\operatorname{emb}(t)_{2i+1} = \cos\left(\frac{t}{10000^{2i/d}}\right)
$$

for `i = 0, ..., d/2 - 1`. This gives each timestep a smooth vector representation containing both low-frequency and high-frequency components.

**Suggested starter architecture**

The following compact U-Net is a good initial starting point for MNIST. It is intentionally small enough to run comfortably during development, and you can later tune the number of channels, number of blocks, normalization layers, or attention layers for better performance.

```text
Input:                 1 x 32 x 32
Encoder block 1:      32 x 32 x 32
Downsample
Encoder block 2:      64 x 16 x 16
Downsample
Bottleneck:          128 x  8 x  8
Upsample + skip 2:    64 x 16 x 16
Upsample + skip 1:    32 x 32 x 32
Output:                1 x 32 x 32
```

A typical implementation would use convolutional blocks such as `Conv2d -> activation -> Conv2d -> activation`, add a projected time embedding into each block, use strided convolutions or pooling for downsampling, and use transposed convolutions or interpolation for upsampling.

**Your Task:**
Implement a U-Net model as a `torch.nn.Module`. The architecture should include:
*   **Sinusoidal Time Embeddings:** A mechanism to convert the integer timestep `t` into a meaningful vector representation that the model can use.
*   **A Downsampling Path (Encoder):** A series of convolutional blocks that decrease the spatial dimensions of the image while increasing the number of channels, capturing contextual information.
*   **A Bottleneck:** The middle part of the U-Net connecting the encoder and decoder.
*   **An Upsampling Path (Decoder):** A series of blocks that increase the spatial dimensions and use skip connections to merge features from the corresponding encoder blocks. This allows the model to use both high-level context and low-level details to make its prediction.
*   **A Final Output Layer:** A final convolution that maps the feature representation back to an image with the same number of channels as the input (1 for MNIST), representing the predicted noise.

Your model should accept noisy images of shape `(batch, 1, 32, 32)` and timesteps of shape `(batch,)`, and return a predicted noise tensor with the same shape as the input images.

This is the most complex part of the implementation. You may use the compact U-Net above; the goal is correctness and clear tensor handling before making the model larger.


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # TODO: convert integer timesteps into sinusoidal embeddings
        return ...


class UNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=32, time_dim=128):
        super().__init__()

        # TODO: define time embedding, downsampling path, bottleneck, upsampling path, and output layer
        pass

    def forward(self, x, t):
        # TODO: predict the noise in x at timestep t
        return ...


# TODO: instantiate the model and verify that one mini-batch passes through it
model = ...


## 5. The Training Loop


This is where we teach our U-Net to denoise. The training objective is to minimize the difference between the noise predicted by our U-Net and the actual noise that was added. We will use the **Mean Squared Error (MSE)** as our loss function.

**Your Task:**
1.  Instantiate your U-Net model and move it to the correct `DEVICE`.
2.  Define your loss function (`nn.MSELoss`) and your optimizer (`torch.optim.Adam`).
3.  Write the main training loop. For each epoch, it should iterate through the `train_loader`. Inside the loop for each batch, you must perform these steps:
    *   Sample a random timestep `t` for each image in the batch.
    *   Use your `forward_diffusion_sample` function to get the `noisy_images` (`x_t`) and the ground-truth `noise` (`epsilon`).
    *   Get the `predicted_noise` by passing the `noisy_images` and timesteps `t` to your U-Net model.
    *   Calculate the MSE `loss` between the `predicted_noise` and the actual `noise`.
    *   Perform the standard backpropagation and optimizer step (`optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`).
4.  Print the loss periodically to monitor training progress.


In [ ]:
model = UNet().to(DEVICE)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (images, _) in enumerate(train_loader):
        images = images.to(DEVICE)

        # TODO: sample one random timestep per image
        t = ...

        # TODO: create noisy images and predict the added noise
        noisy_images, noise = ...
        predicted_noise = ...

        # TODO: compute loss and update model parameters
        loss = ...

        ...

        epoch_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch + 1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    print(f"Epoch {epoch + 1} average loss: {epoch_loss / len(train_loader):.4f}")


## 6. Sampling (Image Generation)


With a trained model, we can finally generate new images! This is done by simulating the **reverse process**. We start with pure noise and iteratively denoise it using our model.

The process is as follows:
1.  Start with an image of pure, random Gaussian noise `x_T`.
2.  Iterate backwards from `t = T-1` down to `0`.
3.  In each step, use the trained U-Net to predict the noise in the current image `x_t`.
4.  Use this prediction and the diffusion formulas to mathematically take a small step backwards, removing a bit of the predicted noise to estimate the slightly cleaner image `x_{t-1}`. 
This can be expressed mathematically as:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z$$

where $\epsilon_\theta(x_t, t)$ is the noise predicted by our U-Net, $z$ is additional random noise, and $\sigma_t$ is the noise scale for the reverse step. For this assignment, set `z = 0` for deterministic sampling.

**Your Task:**
Write a sampling function that takes your trained model as input and generates a specified number of images.
1.  Start by creating a tensor of random noise.
2.  Loop backwards from `TIMESTEPS - 1` down to `0`.
3.  Inside the loop, use your model to predict the noise for the current image `x_t`.
4.  Apply the reverse diffusion formula to calculate `x_{t-1}` from `x_t` and the predicted noise. At `t = 0`, return the final estimate instead of adding another reverse-noise step.
5.  After the loop finishes, the final `x_0` is your generated image.
6.  Generate a batch of 16 images and use your visualization function to display them.


In [ ]:
@torch.no_grad()
def sample(model, num_images=16):
    model.eval()

    # TODO: start from pure Gaussian noise
    x = ...

    for step in reversed(range(TIMESTEPS)):
        t = torch.full((num_images,), step, device=DEVICE, dtype=torch.long)

        # TODO: predict noise and apply one deterministic reverse step
        predicted_noise = ...
        x = ...

    return x


# TODO: generate and visualize samples
generated_images = sample(model, num_images=16)
show_images(generated_images.cpu(), title="Generated samples", nrow=4)
